In [4]:
from pathlib import Path
from typing import Any
import math
import numbers
import pandas as pd

# ============================================================
# FILE CONFIGURATION
# ============================================================

InputFilePath = Path(
    r"C:\Users\StevenFoster\Downloads"
    r"\Project and Type - Operational Excellence Metric Thresholds.xlsx"
)

OutputFileName = "ARG_Operational_Metric_DAX.txt"

CopyToClipboard = True
GenerateLayoutTable = True
BlankStatusWhenValueIsBlank = True
BaseIndent = 0

# ============================================================
# PROJECT TYPE
# ============================================================


def GetProjectTypeFromOutputFileName(OutputFileName: str) -> str:
    """
    Returns the filename prefix before the first underscore.
    """

    ProjectType = OutputFileName.split("_", 1)[0].strip().upper()

    if not ProjectType:
        raise ValueError("Unable to determine ProjectType " "from OutputFileName.")

    return ProjectType


ProjectType = GetProjectTypeFromOutputFileName(OutputFileName)

# ============================================================
# LAYOUT TABLE NAME
# ============================================================

LayoutTableName = f"{ProjectType} Operational Metric Layout"

# ============================================================
# PROJECT CONFIGURATION
# ============================================================

ConfigurationDf = pd.read_excel(
    InputFilePath, sheet_name="Project Configuration", engine="openpyxl"
)

ConfigurationRow = ConfigurationDf[
    ConfigurationDf["ProjectType"].astype(str).str.upper().eq(ProjectType)
]

if ConfigurationRow.empty:
    raise ValueError(
        f"No Project Configuration found " f"for ProjectType '{ProjectType}'."
    )

ProjectConfiguration = ConfigurationRow.iloc[0]

SupportsYoYTrend = str(ProjectConfiguration["SupportsYoY"]).strip().lower() == "yes"

IncludeCommentPanel = (
    str(ProjectConfiguration["IncludeCommentPanel"]).strip().lower() == "yes"
)

TrendingLabel = "YoY Trend" if SupportsYoYTrend else " "

# ============================================================
# METRIC DEFINITIONS
# ============================================================

MetricsDf = pd.read_excel(InputFilePath, sheet_name="Measures", engine="openpyxl")

MetricsDf = MetricsDf[MetricsDf["ProjectType"].astype(str).str.upper().eq(ProjectType)]

if MetricsDf.empty:
    raise ValueError(f"No metrics found for " f"ProjectType '{ProjectType}'.")

Metrics = MetricsDf.to_dict(orient="records")
# ============================================================
# VALIDATION SETTINGS
# ============================================================

ValidOperators = {"<", "<=", ">", ">=", "=", "<>"}
RequiredFields = {
    "SortOrder",
    "MetricName",
    "MetricKey",
    "FormatString",
    "ThresholdFlag",
    "RedOperator",
    "RedThreshold",
    "AmberOperator",
    "AmberThreshold",
    "YTDFlag",
    "YoYFlag",
    "YTDTargetFlag",
    "MonthTargetFlag",
    "BenchmarkName",
    "Measure",
    "Target",
}

# ============================================================
# HELPER FUNCTIONS
# ============================================================


def IsBenchmarkMeasure(MeasureReference: str) -> bool:
    """
    Determines whether a measure should use the
    benchmark-aware functions.
    """
    return MeasureReference in {"[Actuals]", "[Target]"}


def HasTextValue(Value: Any) -> bool:
    """Returns True when a metadata value is not blank or NaN."""
    return pd.notna(Value) and bool(str(Value).strip())


def NormalizeWholeNumber(Value: Any, FieldName: str, MetricName: str) -> int:
    """Converts Excel numeric values or numeric strings to a whole number."""
    if isinstance(Value, bool) or not HasTextValue(Value):
        raise TypeError(f"{FieldName} for '{MetricName}' must be a whole number.")

    try:
        NumericValue = float(str(Value).strip())
    except (TypeError, ValueError) as ConversionError:
        raise TypeError(
            f"{FieldName} for '{MetricName}' must be numeric. " f"Received: {Value!r}."
        ) from ConversionError

    if not math.isfinite(NumericValue) or not NumericValue.is_integer():
        raise TypeError(
            f"{FieldName} for '{MetricName}' must be a whole number. "
            f"Received: {Value!r}."
        )

    return int(NumericValue)


def NormalizeNumber(Value: Any, FieldName: str, MetricName: str) -> int | float:
    """Converts Excel numeric values or numeric strings to a finite number."""
    if isinstance(Value, bool) or not HasTextValue(Value):
        raise TypeError(f"{FieldName} for '{MetricName}' must be numeric.")

    try:
        NumericValue = float(str(Value).strip())
    except (TypeError, ValueError) as ConversionError:
        raise TypeError(
            f"{FieldName} for '{MetricName}' must be numeric. " f"Received: {Value!r}."
        ) from ConversionError

    if not math.isfinite(NumericValue):
        raise TypeError(f"{FieldName} for '{MetricName}' must be a finite number.")

    return int(NumericValue) if NumericValue.is_integer() else NumericValue


def ValidateMetrics(MetricList: list[dict[str, Any]]) -> None:
    """Validates and normalizes metric metadata before generating DAX."""
    if not MetricList:
        raise ValueError("Metrics cannot be empty.")

    MetricKeys: list[str] = []
    ProjectSortOrders: list[tuple[str, str, int]] = []

    for MetricIndex, Metric in enumerate(MetricList, start=1):
        MissingFields = RequiredFields.difference(Metric.keys())

        if MissingFields:
            MissingText = ", ".join(sorted(MissingFields))
            raise ValueError(
                f"Metric {MetricIndex} is missing required fields: {MissingText}"
            )

        MetricName = str(Metric["MetricName"]).strip()
        MetricKey = str(Metric["MetricKey"]).strip()
        FormatString = str(Metric["FormatString"]).strip()

        if not MetricName:
            raise ValueError(f"Metric {MetricIndex} has a blank MetricName.")

        if not MetricKey:
            raise ValueError(f"Metric {MetricIndex} has a blank MetricKey.")

        if not MetricKey.replace("_", "").isalnum():
            raise ValueError(
                f"MetricKey '{MetricKey}' contains unsupported characters. "
                "Use letters, numbers, or underscores only."
            )

        if not FormatString:
            raise ValueError(f"Metric '{MetricName}' has a blank FormatString.")

        if not ProjectType == "ARG":
            for FlagName in (
                "ThresholdFlag",
                "YTDFlag",
                "YoYFlag",
                "YTDTargetFlag",
                "MonthTargetFlag",
                "MonthTargetDisplayFlag",
            ):
                FlagValue = str(Metric[FlagName]).strip().lower()

                if FlagValue not in {"yes", "no"}:
                    raise ValueError(
                        f"Metric '{MetricName}' has an invalid {FlagName}. "
                        "Use Yes or No."
                    )

                Metric[FlagName] = FlagValue

        ThresholdFlag = Metric["ThresholdFlag"]
        RedOperator = str(Metric["RedOperator"]).strip()
        AmberOperator = str(Metric["AmberOperator"]).strip()

        if ThresholdFlag == "yes":
            if RedOperator not in ValidOperators:
                raise ValueError(
                    f"Metric '{MetricName}' has unsupported RedOperator "
                    f"'{RedOperator}'."
                )

            if AmberOperator not in ValidOperators:
                raise ValueError(
                    f"Metric '{MetricName}' has unsupported AmberOperator "
                    f"'{AmberOperator}'."
                )

            Metric["RedThreshold"] = NormalizeNumber(
                Metric["RedThreshold"], "RedThreshold", MetricName
            )
            Metric["AmberThreshold"] = NormalizeNumber(
                Metric["AmberThreshold"], "AmberThreshold", MetricName
            )

        Metric["SortOrder"] = NormalizeWholeNumber(
            Metric["SortOrder"], "SortOrder", MetricName
        )

        if Metric["YTDTargetFlag"] == "yes" and not HasTextValue(Metric["Target"]):
            raise ValueError(
                f"Metric '{MetricName}' has YTDTargetFlag set to Yes "
                "but Target is blank."
            )

        if Metric["MonthTargetFlag"] == "yes" and not HasTextValue(Metric["Target"]):
            raise ValueError(
                f"Metric '{MetricName}' has MonthTargetFlag set to Yes "
                "but Target is blank."
            )

        MonthTargetDisplayFlag = Metric["MonthTargetDisplayFlag"]
        MonthTargetFlag = Metric["MonthTargetFlag"]

        if MonthTargetDisplayFlag == "yes" and MonthTargetFlag != "yes":
            raise ValueError(
                f"Metric '{MetricName}' has MonthTargetDisplayFlag set to Yes "
                "but MonthTargetFlag is not set to Yes."
            )

        if MonthTargetDisplayFlag == "yes" and not HasTextValue(Metric["Target"]):
            raise ValueError(
                f"Metric '{MetricName}' has MonthTargetDisplayFlag set to Yes "
                "but Target is blank."
            )

        MetricKeys.append(MetricKey)
        ProjectSortOrders.append(
            (
                str(Metric["ProjectType"]).strip().upper(),
                str(Metric["ProjectName"]).strip().upper(),
                Metric["SortOrder"],
            )
        )

    DuplicateMetricKeys = {
        MetricKey for MetricKey in MetricKeys if MetricKeys.count(MetricKey) > 1
    }
    if DuplicateMetricKeys:
        DuplicateText = ", ".join(sorted(DuplicateMetricKeys))
        raise ValueError(f"Duplicate MetricKey values found: {DuplicateText}")

    DuplicateProjectSortOrders = {
        ProjectSortOrder
        for ProjectSortOrder in ProjectSortOrders
        if ProjectSortOrders.count(ProjectSortOrder) > 1
    }
    if DuplicateProjectSortOrders:

        DuplicateText = ", ".join(
            (f"{ProjectType} | " f"{ProjectName} | " f"{SortOrder}")
            for ProjectType, ProjectName, SortOrder in sorted(
                DuplicateProjectSortOrders
            )
        )

        raise ValueError(
            "Duplicate SortOrder values found "
            "within the same project layout: "
            f"{DuplicateText}"
        )


def GetVariableName(MetricKey: str) -> str:
    """Converts a metadata key to the requested Power BI variable pattern."""
    CleanMetricKey = str(MetricKey).strip()
    if not CleanMetricKey:
        raise ValueError("A variable name cannot be blank.")
    return CleanMetricKey[0].lower() + CleanMetricKey[1:]


def GetMeasureReference(MeasureName: Any) -> str:
    """Returns a DAX measure reference, adding brackets when needed."""
    CleanMeasureName = str(MeasureName).strip()
    if not CleanMeasureName:
        raise ValueError("A measure name cannot be blank.")
    if CleanMeasureName.startswith("[") and CleanMeasureName.endswith("]"):
        return CleanMeasureName
    return f"[{CleanMeasureName}]"


def EscapeDaxText(Value: Any) -> str:
    return str(Value).replace('"', '""')


def FormatDaxNumber(Value: int | float) -> str:
    if isinstance(Value, bool) or not isinstance(Value, numbers.Real):
        raise TypeError("Threshold values must be numeric.")
    return format(Value, ".15g")


def GetIndentedText(Text: str, IndentLevel: int) -> str:
    Prefix = " " * IndentLevel
    return "\n".join(f"{Prefix}{Line}" if Line else "" for Line in Text.splitlines())


def BuildSectionHeader(SectionName: str) -> str:
    return (
        "/* ============================================================\n"
        f"   {SectionName}\n"
        "   ============================================================ */"
    )


def BuildWioaCommentSection() -> str:
    return """
/* ============================================================
    OPERATIONAL EXCELLENCE COMMENTS TABLE
============================================================ */

VAR __adult1 =
        FosterBI.YTDValue([Adult New Enrollments], [Current Fiscal Year],[Current Month Sort] ) 
VAR __dw1 =
    FosterBI.YTDValue([DW New Enrollments], [Current Fiscal Year],[Current Month Sort] ) 
VAR __adult1Target =
    FosterBI.YTDValue([Adult New Enrollments Target], [Current Fiscal Year],[Current Month Sort] ) 
VAR __dw1Target =
    FosterBI.YTDValue([DW New Enrollments Target], [Current Fiscal Year],[Current Month Sort] ) 
VAR __youth1 =
        FosterBI.YTDValue([Youth New Enrollments], [Current Fiscal Year],[Current Month Sort] ) 
VAR __youth1Target =
    FosterBI.YTDValue([Youth New Enrollments Target], [Current Fiscal Year],[Current Month Sort] ) 

VAR _adult2 = BLANK ( )
VAR _dw2 = BLANK ( )
VAR _youth2 = BLANK ( )
VAR __adultQ4 =
    FosterBI.MonthValue([AD- Placement on unsubsidized employment Q4 after exit], [Month Year Current])
VAR __adultQ4Target =
    FosterBI.MonthValue([AD- Placement on unsubsidized employment Q4 after exit Target], [Month Year Current])
VAR __adultQ4Display =
    FORMAT
    (
        DIVIDE ( __adultQ4, 100 ),
        "#,##0%"
    )
        & IF
        (
            ISBLANK
            (
                DIVIDE ( __adultQ4, __adultQ4Target )
            ),
            "",
            " ("
                & FORMAT
                (
                    DIVIDE ( __adultQ4, __adultQ4Target ),
                    "0%"
                )
                & ")"
        )
VAR __dwQ4 =
    FosterBI.MonthValue([DW- Placement on unsubsidized employment Q4 after exit], [Month Year Current])
VAR __dwQ4Target =
    FosterBI.MonthValue([DW- Placement on unsubsidized employment Q4 after exit Target], [Month Year Current])
VAR __dwQ4Display =
    FORMAT
    (
        DIVIDE ( __dwQ4, 100 ),
        "#,##0%"
    )
        & IF
        (
            ISBLANK
            (
                DIVIDE ( __dwQ4, __dwQ4Target )
            ),
            "",
            " ("
                & FORMAT
                (
                    DIVIDE ( __dwQ4, __dwQ4Target ),
                    "0%"
                )
                & ")"
        )
VAR __youthQ4 =
    FosterBI.MonthValue([Placement on Education/training activities, or unsubsidized employment  Q4], [Month Year Current])
VAR __youthQ4Target =
    FosterBI.MonthValue([Placement on Education/training activities, or unsubsidized employment  Q4 Target], [Month Year Current])
VAR __youthQ4Display =
    FORMAT
    (
        DIVIDE ( __youthQ4, 100 ),
        "#,##0%"
    )
        & IF
        (
            ISBLANK
            (
                DIVIDE ( __youthQ4, __youthQ4Target )
            ),
            "",
            " ("
                & FORMAT
                (
                    DIVIDE ( __youthQ4, __youthQ4Target ),
                    "0%"
                )
                & ")"
        )
"""


def BuildWioaCommentStatus() -> str:
    return """
/* ============================================================
    OPERATIONAL EXCELLENCE COMMENTS STATUS
============================================================ */

VAR __totalEnrollActual =
    COALESCE ( __adult1, 0 )
        + COALESCE ( __dw1, 0 )
        + COALESCE ( __youth1, 0 )
VAR __totalEnrollTarget =
    COALESCE ( __adult1Target, 0 )
        + COALESCE ( __dw1Target, 0 )
        + COALESCE ( __youth1Target, 0 )
VAR __overallEnrollStatus =
    SWITCH
    (
        TRUE ( ),
        __totalEnrollTarget = 0, BLANK ( ),
        DIVIDE
        (
            __totalEnrollActual,
            __totalEnrollTarget
        )
            < 0.95, "red",
        DIVIDE
        (
            __totalEnrollActual,
            __totalEnrollTarget
        )
            < 1, "amber",
        "green"
    )
VAR __totalQ4Actual =
    COALESCE ( __adultQ4, 0 )
        + COALESCE ( __dwQ4, 0 )
        + COALESCE ( __youthQ4, 0 )
VAR __totalQ4Target =
    COALESCE ( __adultQ4Target, 0 )
        + COALESCE ( __dwQ4Target, 0 )
        + COALESCE ( __youthQ4Target, 0 )
VAR __overallQ4Status =
    SWITCH
    (
        TRUE ( ),
        __totalQ4Target = 0, BLANK ( ),
        DIVIDE
        (
            __totalQ4Actual,
            __totalQ4Target
        )
            < 0.95, "red",
        DIVIDE
        (
            __totalQ4Actual,
            __totalQ4Target
        )
            < 1, "amber",
        "green"
    )
"""


def BuildWioaPerformanceTable() -> str:
    return """
/* ============================================================
    OPERATIONAL EXCELLENCE COMMENTS HTML
============================================================  */
VAR _performanceTable =
    VAR _row1 =
        "<tr>
            <td>Total Enrollments</td>
            <td>"
            & "PY '" & FORMAT ( RIGHT ( [Current Fiscal Year], 2 ), "##" )
            & "</td>
            <td>"
            & FORMAT ( __adult1, "#,##0" )
            & "</td>
            <td>"
            & FORMAT ( __dw1, "#,##0" )
            & "</td>
            <td>"
            & FORMAT ( __youth1, "#,##0" )
            & "</td>
            <td class='status'>
                Overall Status:
                <span class='dot "
            & COALESCE ( __overallEnrollStatus, "" )
            & "'></span>
            </td>
        </tr>"
    VAR _row2 =
        "<tr>
            <td>Real-Time Credential</td>
            <td>"
            & "PY '" & FORMAT ( RIGHT ( [Current Fiscal Year], 2 ), "##" )
            & "</td>
            <td>"
            & FORMAT ( _adult2, "#,##0" )
            & "</td>
            <td>"
            & FORMAT ( _dw2, "#,##0" )
            & "</td>
            <td>"
            & FORMAT ( _youth2, "#,##0" )
            & "</td>
            <td><span class='dot'></span></td>
        </tr>"
    VAR _row3 =
        "<tr>
            <td>Q4 % Employment (% Target)</td>
            <td>"
            & [Month Display Current]
            & "</td>
            <td>"
            & FORMAT ( _adult2, "#,##0" )
            & "</td>
            <td>"
            & FORMAT ( _dw2, "#,##0" )
            & "</td>
            <td>"
            & FORMAT ( _youth2, "#,##0" )
            & "</td>
            <td class='status'>
                Overall Status:
                <span class='dot "
            & COALESCE ( __overallQ4Status, "" )
            & "'></span>
            </td>
        </tr>"
    RETURN
        "
    <table class='commentaryTable'>
        <thead>
            <tr>
                <th>Measure</th>
                <th>Period</th>
                <th>Adult</th>
                <th>DW</th>
                <th>Youth</th>
                <th> </th>
            </tr>
        </thead>
        <tbody>
            "
            & _row1
            & _row2
            & _row3
            & "
        </tbody>
    </table>
    "
"""


# ============================================================
# DAX LAYOUT TABLE GENERATOR
# ============================================================


def BuildLayoutTable(TableName: str, MetricList: list[dict[str, Any]]) -> str:
    """
    Creates the Project-aware DATATABLE used by the
    Operational Excellence measure.
    """

    SortedMetrics = sorted(
        MetricList, key=lambda Metric: (Metric["ProjectName"], Metric["SortOrder"])
    )

    TableRows = []

    for Metric in SortedMetrics:

        ProjectName = EscapeDaxText(
                str(Metric["ProjectName"])
                .strip()
                .upper()
            )

        MetricName = EscapeDaxText(Metric["MetricName"])

        MetricKey = EscapeDaxText(Metric["MetricKey"])

        SortOrder = Metric["SortOrder"]

        TableRows.append(
            f'    {{"{ProjectName}", {SortOrder}, "{MetricName}", "{MetricKey}"}}'
        )

    RowText = ",\n".join(TableRows)

    return (
        f"{TableName} =\n"
        "DATATABLE(\n"
        '    "ProjectName", STRING,\n'
        '    "SortOrder", INTEGER,\n'
        '    "MetricName", STRING,\n'
        '    "MetricKey", STRING,\n'
        "{\n"
        f"{RowText}\n"
        "}\n"
        ")"
    )


# ============================================================
# FUNCTION BUILDER
# ============================================================
def BuildMonthValueCall(
    MeasureReference: str, BenchmarkName: str, PeriodReference: str
) -> list[str]:

    if IsBenchmarkMeasure(MeasureReference):
        return [
            "                        MonthValue(",
            f"                            {MeasureReference},",
            f'                            "{EscapeDaxText(BenchmarkName)}",',
            f"                            {PeriodReference}",
            "                        )",
        ]

    return [
        "                        FosterBI.MonthValue(",
        f"                            {MeasureReference},",
        f"                            {PeriodReference}",
        "                        )",
    ]


def BuildYTDValueCall(MeasureReference: str, BenchmarkName: str) -> list[str]:

    if IsBenchmarkMeasure(MeasureReference):
        return [
            "                        YTDValue(",
            f"                            {MeasureReference},",
            f'                            "{EscapeDaxText(BenchmarkName)}",',
            "                            [Current Fiscal Year],",
            "                            [Current Month Sort]",
            "                        )",
        ]

    return [
        "                        FosterBI.YTDValue(",
        f"                            {MeasureReference},",
        "                            [Current Fiscal Year],",
        "                            [Current Month Sort]",
        "                        )",
    ]


def BuildPYTDValueCall(MeasureReference: str, BenchmarkName: str) -> list[str]:

    if IsBenchmarkMeasure(MeasureReference):
        return [
            "                        PYTDValue(",
            f"                            {MeasureReference},",
            f'                            "{EscapeDaxText(BenchmarkName)}",',
            "                            [Current Fiscal Year],",
            "                            [Prior Fiscal YTD]",
            "                        )",
        ]

    return [
        "                        FosterBI.PYTDValue(",
        f"                            {MeasureReference},",
        "                            [Current Fiscal Year],",
        "                            [Prior Fiscal YTD]",
        "                        )",
    ]


# ============================================================
# PLACEHOLDER VARIABLE GENERATOR
# ============================================================


def BuildPlaceholderVariables(MetricList: list[dict[str, Any]]) -> str:
    PlaceholderBlocks = [BuildSectionHeader("OPERATIONAL EXCELLENCE PLACEHOLDERS")]

    for Index, Metric in enumerate(MetricList):
        VariableName = GetVariableName(Metric["MetricKey"])
        MeasureReference = GetMeasureReference(Metric["Measure"])
        YtdFlag = str(Metric["YTDFlag"]).strip().lower() == "yes"
        YoyFlag = str(Metric["YoYFlag"]).strip().lower() == "yes"
        YtdTargetFlag = str(Metric["YTDTargetFlag"]).strip().lower() == "yes"
        MonthTargetFlag = str(Metric["MonthTargetFlag"]).strip().lower() == "yes"
        IsDaysAheadMetric = (
            ProjectType == "ARG"
            and "days ahead of target" in Metric["MetricName"].lower()
        )

        PreviousMetric = MetricList[Index - 1] if Index > 0 else None

        TargetReference = (
            GetMeasureReference(Metric["Target"])
            if HasTextValue(Metric["Target"])
            else ""
        )
        if IsDaysAheadMetric:
            if PreviousMetric is None:
                raise ValueError(
                    f"'{Metric['MetricName']}' cannot reference a previous row."
                )

            PreviousVariableName = GetVariableName(PreviousMetric["MetricKey"])

            VariableLines = [
                f"VAR _{VariableName}Month3 = "
                f"_{PreviousVariableName}TargetMonth3 - "
                f"_{PreviousVariableName}Month3",
                f"VAR _{VariableName}Month2 = "
                f"_{PreviousVariableName}TargetMonth2 - "
                f"_{PreviousVariableName}Month2",
                f"VAR _{VariableName}Month1 = "
                f"_{PreviousVariableName}TargetMonth1 - "
                f"_{PreviousVariableName}Month1",
            ]

            if YtdFlag:
                VariableLines.append(
                    f"VAR _{VariableName}Ytd = "
                    f"_{PreviousVariableName}Ytd - "
                    f"_{PreviousVariableName}Target"
                )
        else:
            VariableLines = [
                (
                    f"VAR _{VariableName}Month3 = "
                    f"FosterBI.MonthValue({MeasureReference}, [Month Year Prior 2]) "
                ),
                (
                    f"VAR _{VariableName}Month2 = "
                    f"FosterBI.MonthValue({MeasureReference}, [Month Year Prior 1]) "
                ),
                (
                    f"VAR _{VariableName}Month1 = "
                    f"FosterBI.MonthValue({MeasureReference}, [Month Year Current]) "
                ),
            ]

        if YtdFlag:
            VariableLines.append(
                f"VAR _{VariableName}Ytd = "
                f"FosterBI.YTDValue({MeasureReference}, [Current Fiscal Year],[Current Month Sort] ) "
            )

        if YtdTargetFlag:
            VariableLines.append(
                f"VAR _{VariableName}Target = "
                f"FosterBI.YTDValue({TargetReference}, [Current Fiscal Year],[Current Month Sort] ) "
            )

        if YtdFlag and YtdTargetFlag:
            VariableLines.append(
                f"VAR _{VariableName}Display = "
                f'FORMAT(_{VariableName}Ytd, "#,##") '
                f"& IF(ISBLANK(DIVIDE(_{VariableName}Ytd, _{VariableName}Target)), "
                f'"", " (" & FORMAT(DIVIDE(_{VariableName}Ytd, '
                f'_{VariableName}Target), "0%") & ")")'
            )

        if YoyFlag:
            if not YtdFlag:
                raise ValueError(
                    f"Metric '{Metric['MetricName']}' has YoYFlag set to Yes "
                    "but YTDFlag is No."
                )
            VariableLines.extend(
                [
                    (
                        f"VAR _{VariableName}Pytd = "
                        f"FosterBI.PYTDValue({MeasureReference}, [Current Fiscal Year],[Prior Fiscal YTD] )"
                    ),
                    f"VAR _{VariableName}Var = _{VariableName}Ytd - _{VariableName}Pytd",
                    (
                        f"VAR _{VariableName}VariancePct = "
                        f"DIVIDE(_{VariableName}Var, _{VariableName}Pytd)"
                    ),
                    (
                        f"VAR _{VariableName}Trend = "
                        f"SWITCH(TRUE(), ISBLANK(_{VariableName}Var), BLANK(), "
                        f'_{VariableName}Var > 0, "up", '
                        f'_{VariableName}Var < 0, "down", BLANK())'
                    ),
                ]
            )

        if MonthTargetFlag:
            MetricFormatString = EscapeDaxText(Metric["FormatString"])
            VariableLines.extend(
                [
                    (
                        f"VAR _{VariableName}TargetMonth3 = "
                        f"FosterBI.MonthValue({TargetReference}, [Month Year Prior 2]) "
                    ),
                    (
                        f"VAR _{VariableName}TargetMonth2 = "
                        f"FosterBI.MonthValue({TargetReference}, [Month Year Prior 1]) "
                    ),
                    (
                        f"VAR _{VariableName}TargetMonth1 = "
                        f"FosterBI.MonthValue({TargetReference}, [Month Year Current]) "
                    ),
                    (
                        f"VAR _{VariableName}Month3Display = "
                        f'FORMAT(DIVIDE(_{VariableName}Month3,100), "{MetricFormatString}") '
                        f"& IF(ISBLANK(DIVIDE(_{VariableName}Month3, "
                        f'_{VariableName}TargetMonth3)), "", " (" & '
                        f"FORMAT(DIVIDE(_{VariableName}Month3, "
                        f'_{VariableName}TargetMonth3), "0%") & ")")'
                    ),
                    (
                        f"VAR _{VariableName}Month2Display = "
                        f'FORMAT(DIVIDE(_{VariableName}Month2,100), "{MetricFormatString}") '
                        f"& IF(ISBLANK(DIVIDE(_{VariableName}Month2, "
                        f'_{VariableName}TargetMonth2)), "", " (" & '
                        f"FORMAT(DIVIDE(_{VariableName}Month2, "
                        f'_{VariableName}TargetMonth2), "0%") & ")")'
                    ),
                    (
                        f"VAR _{VariableName}Month1Display = "
                        f'FORMAT(DIVIDE(_{VariableName}Month1,100), "{MetricFormatString}") '
                        f"& IF(ISBLANK(DIVIDE(_{VariableName}Month1, "
                        f'_{VariableName}TargetMonth1)), "", " (" & '
                        f"FORMAT(DIVIDE(_{VariableName}Month1, "
                        f'_{VariableName}TargetMonth1), "0%") & ")")'
                    ),
                ]
            )

        PlaceholderBlocks.append("\n".join(VariableLines))

    return "\n\n".join(PlaceholderBlocks)


# ============================================================
# STATUS VARIABLE GENERATOR
# ============================================================


def BuildStatusVariables(
    MetricList: list[dict[str, Any]], ReturnBlankStatus: bool
) -> str:
    StatusBlocks = [BuildSectionHeader("OPERATIONAL EXCELLENCE STATUS")]

    for Metric in MetricList:
        VariableName = GetVariableName(Metric["MetricKey"])

        if Metric["ThresholdFlag"] == "no":
            StatusBlocks.append(f"VAR _{VariableName}Status = BLANK()")
            continue

        RedOperator = str(Metric["RedOperator"]).strip()
        RedThreshold = FormatDaxNumber(Metric["RedThreshold"])
        AmberOperator = str(Metric["AmberOperator"]).strip()
        AmberThreshold = FormatDaxNumber(Metric["AmberThreshold"])

        YtdTargetFlag = str(Metric.get("YTDTargetFlag", "no")).lower()
        MonthTargetFlag = str(Metric.get("MonthTargetFlag", "no")).lower()

        # Determine which expression to evaluate
        if YtdTargetFlag == "yes":
            ValueExpression = f"DIVIDE(_{VariableName}Ytd, _{VariableName}Target)"
            BlankCheck = (
                f"ISBLANK(_{VariableName}Ytd) || " f"ISBLANK(_{VariableName}Target)"
            )
        elif MonthTargetFlag == "yes":
            ValueExpression = (
                f"DIVIDE(_{VariableName}Month1, _{VariableName}TargetMonth1)"
            )
            BlankCheck = (
                f"ISBLANK(_{VariableName}Month1) || "
                f"ISBLANK(_{VariableName}TargetMonth1)"
            )
        else:
            StatusBlocks.append(f"VAR _{VariableName}Status = BLANK()")
            continue

        StatusLines = [
            f"VAR _{VariableName}Status =",
            "    SWITCH(",
            "        TRUE(),",
        ]

        if ReturnBlankStatus:
            StatusLines.append(f"        {BlankCheck}, BLANK(),")

        StatusLines.extend(
            [
                f'        {ValueExpression} {RedOperator} {RedThreshold}, "red",',
                f'        {ValueExpression} {AmberOperator} {AmberThreshold}, "amber",',
                '        "green"',
                "    )",
            ]
        )

        StatusBlocks.append("\n".join(StatusLines))

    return "\n\n".join(StatusBlocks)


# ============================================================
# SWITCH GENERATORS
# ============================================================


def BuildMetricSwitch(
    VariableName: str,
    MetricList: list[dict[str, Any]],
    ValueSuffix: str | None = None,
    ValueSuffixField: str | None = None,
    IndentLevel: int = 4,
) -> str:
    if ValueSuffix is None and ValueSuffixField is None:
        raise ValueError("ValueSuffix or ValueSuffixField must be provided.")

    Indent = " " * IndentLevel
    InnerIndent = " " * (IndentLevel + 4)
    ValueIndent = " " * (IndentLevel + 8)
    SwitchLines = [
        f"{Indent}VAR _{VariableName} =",
        f"{InnerIndent}SWITCH(",
        f"{ValueIndent}_metricKey,",
    ]

    MetricLines = []
    for Metric in MetricList:
        MetricKeyText = EscapeDaxText(Metric["MetricKey"])
        MetricVariable = GetVariableName(Metric["MetricKey"])
        Suffix = Metric.get(ValueSuffixField) if ValueSuffixField else ValueSuffix

        if not HasTextValue(Suffix):
            raise ValueError(
                f"Metric '{Metric['MetricName']}' does not have a valid "
                f"suffix for {VariableName}."
            )

        MetricLines.append(f'{ValueIndent}"{MetricKeyText}", _{MetricVariable}{Suffix}')

    for MetricIndex, MetricLine in enumerate(MetricLines):
        LineEnding = "" if MetricIndex == len(MetricLines) - 1 else ","
        SwitchLines.append(f"{MetricLine}{LineEnding}")

    SwitchLines.append(f"{InnerIndent})")
    return "\n".join(SwitchLines)


def BuildFormatSwitch(MetricList: list[dict[str, Any]], IndentLevel: int = 4) -> str:
    Indent = " " * IndentLevel
    InnerIndent = " " * (IndentLevel + 4)
    ValueIndent = " " * (IndentLevel + 8)
    SwitchLines = [
        f"{Indent}VAR _formatString =",
        f"{InnerIndent}SWITCH(",
        f"{ValueIndent}_metricKey,",
    ]

    MetricLines = []
    for Metric in MetricList:
        MetricKey = EscapeDaxText(Metric["MetricKey"])
        FormatString = EscapeDaxText(Metric["FormatString"])
        MetricLines.append(f'{ValueIndent}"{MetricKey}", "{FormatString}"')

    for MetricIndex, MetricLine in enumerate(MetricLines):
        LineEnding = "" if MetricIndex == len(MetricLines) - 1 else ","
        SwitchLines.append(f"{MetricLine}{LineEnding}")

    SwitchLines.append(f"{InnerIndent})")
    return "\n".join(SwitchLines)


# ============================================================
# GENERATED METRIC TABLE
# ============================================================
def IndentDaxLines(DaxLines: list[str], Spaces: int) -> list[str]:
    """Adds indentation to generated DAX lines."""
    Prefix = " " * Spaces
    return [f"{Prefix}{Line}" if Line else "" for Line in DaxLines]


def BuildMetricValueExpression(
    ValueType: str,
    MeasureReference: str,
    BenchmarkName: str,
    PeriodReference: str | None = None,
) -> str:
    """
    Returns one complete Month, YTD, or PYTD DAX expression.
    """

    CleanValueType = str(ValueType).strip().lower()

    if CleanValueType == "month":
        if not HasTextValue(PeriodReference):
            raise ValueError("PeriodReference is required when ValueType is 'month'.")

        ExpressionLines = BuildMonthValueCall(
            MeasureReference=MeasureReference,
            BenchmarkName=BenchmarkName,
            PeriodReference=PeriodReference,
        )

    elif CleanValueType == "ytd":
        ExpressionLines = BuildYTDValueCall(
            MeasureReference=MeasureReference,
            BenchmarkName=BenchmarkName,
        )

    elif CleanValueType == "pytd":
        ExpressionLines = BuildPYTDValueCall(
            MeasureReference=MeasureReference,
            BenchmarkName=BenchmarkName,
        )

    else:
        raise ValueError(
            "Unsupported ValueType "
            f"'{ValueType}'. Expected 'month', 'ytd', or 'pytd'."
        )

    return "\n".join(Line.strip() for Line in ExpressionLines)


def BuildMetricSwitchVariable(
    VariableName: str,
    MetricList: list[dict[str, Any]],
    ExpressionBuilder,
) -> list[str]:
    """
    Builds a generic variable resolved by the current MetricKey.

    If no metrics generate an expression, a simple BLANK() variable
    is generated instead of an empty SWITCH().
    """

    BranchLines = []

    for Index, Metric in enumerate(MetricList):
        Expression = ExpressionBuilder(
            Index,
            Metric,
        )

        if Expression is None:
            continue

        MetricKey = EscapeDaxText(Metric["MetricKey"])

        ExpressionLines = Expression.splitlines()

        BranchLines.append(f'                    "{MetricKey}",')

        BranchLines.extend(
            f"                        {Line}" for Line in ExpressionLines
        )

        BranchLines[-1] = f"{BranchLines[-1]},"

    #
    # No valid expressions
    #
    if not BranchLines:
        return [
            f"            VAR _{VariableName} = BLANK()",
            "",
        ]

    #
    # Generate SWITCH
    #
    Lines = [
        f"            VAR _{VariableName} =",
        "                SWITCH(",
        "                    _metricKey,",
    ]

    Lines.extend(BranchLines)

    Lines.extend(
        [
            "                    BLANK()",
            "                )",
            "",
        ]
    )

    return Lines


# --------------------------------------------------------
# BUILD METRICS TABLE
# --------------------------------------------------------
def BuildMetricTable(
    MetricList: list[dict[str, Any]],
) -> str:
    """
    Builds one virtual _metricTable.

    Each row has the same generic value columns regardless of project or
    metric count. Only metrics in _projectMetrics are iterated.

    MonthTargetDisplayFlag controls whether each metric displays its
    monthly target comparison percentage.
    """

    Lines = [
        "VAR _metricTable =",
        "    GENERATE(",
        "        _projectMetrics,",
        "        VAR _metricKey = [MetricKey]",
        "",
    ]

    def GetBenchmarkName(
        Metric: dict[str, Any],
    ) -> str:
        """
        Returns the metric benchmark name or an empty string.
        """

        Value = Metric.get(
            "BenchmarkName",
            "",
        )

        return str(Value).strip() if HasTextValue(Value) else ""

    def BuildActualMonth(
        PeriodReference: str,
    ):
        """
        Returns a builder for a monthly actual expression.
        """

        def Builder(
            Index: int,
            Metric: dict[str, Any],
        ) -> str:
            MeasureReference = GetMeasureReference(Metric["Measure"])

            BenchmarkName = GetBenchmarkName(Metric)

            IsDaysAheadMetric = (
                ProjectType == "ARG"
                and "days ahead of target" in str(Metric["MetricName"]).strip().lower()
            )

            if IsDaysAheadMetric:
                if Index == 0:
                    raise ValueError(
                        f"'{Metric['MetricName']}' cannot reference "
                        "a previous metric."
                    )

                PreviousMetric = MetricList[Index - 1]

                CurrentProject = str(Metric["ProjectName"]).strip().upper()

                PreviousProject = str(PreviousMetric["ProjectName"]).strip().upper()

                if CurrentProject != PreviousProject:
                    raise ValueError(
                        f"'{Metric['MetricName']}' cannot reference "
                        "a metric from another project."
                    )

                if not HasTextValue(PreviousMetric.get("Target")):
                    raise ValueError(
                        f"'{Metric['MetricName']}' references the "
                        f"previous metric "
                        f"'{PreviousMetric['MetricName']}', but that "
                        "metric does not have a target."
                    )

                PreviousMeasure = GetMeasureReference(PreviousMetric["Measure"])

                PreviousTarget = GetMeasureReference(PreviousMetric["Target"])

                PreviousBenchmark = GetBenchmarkName(PreviousMetric)

                TargetExpression = BuildMetricValueExpression(
                    ValueType="month",
                    MeasureReference=PreviousTarget,
                    BenchmarkName=PreviousBenchmark,
                    PeriodReference=PeriodReference,
                )

                ActualExpression = BuildMetricValueExpression(
                    ValueType="month",
                    MeasureReference=PreviousMeasure,
                    BenchmarkName=PreviousBenchmark,
                    PeriodReference=PeriodReference,
                )

                return f"({TargetExpression}) " f"- ({ActualExpression})"

            return BuildMetricValueExpression(
                ValueType="month",
                MeasureReference=MeasureReference,
                BenchmarkName=BenchmarkName,
                PeriodReference=PeriodReference,
            )

        return Builder

    def BuildTargetMonth(
        PeriodReference: str,
    ):
        """
        Returns a builder for a monthly target expression.
        """

        def Builder(
            Index: int,
            Metric: dict[str, Any],
        ) -> str | None:
            MonthTargetFlag = str(Metric["MonthTargetFlag"]).strip().lower() == "yes"

            if not MonthTargetFlag or not HasTextValue(Metric.get("Target")):
                return None

            return BuildMetricValueExpression(
                ValueType="month",
                MeasureReference=GetMeasureReference(Metric["Target"]),
                BenchmarkName=GetBenchmarkName(Metric),
                PeriodReference=PeriodReference,
            )

        return Builder

    def BuildYtd(
        Index: int,
        Metric: dict[str, Any],
    ) -> str | None:
        """
        Builds a YTD expression when enabled for the metric.
        """

        if str(Metric["YTDFlag"]).strip().lower() != "yes":
            return None

        IsDaysAheadMetric = (
            ProjectType == "ARG"
            and "days ahead of target" in str(Metric["MetricName"]).strip().lower()
        )

        if IsDaysAheadMetric:
            if Index == 0:
                raise ValueError(
                    f"'{Metric['MetricName']}' cannot reference " "a previous metric."
                )

            PreviousMetric = MetricList[Index - 1]

            CurrentProject = str(Metric["ProjectName"]).strip().upper()

            PreviousProject = str(PreviousMetric["ProjectName"]).strip().upper()

            if CurrentProject != PreviousProject:
                raise ValueError(
                    f"'{Metric['MetricName']}' cannot reference "
                    "a metric from another project."
                )

            if not HasTextValue(PreviousMetric.get("Target")):
                raise ValueError(
                    f"'{Metric['MetricName']}' references the "
                    f"previous metric "
                    f"'{PreviousMetric['MetricName']}', but that "
                    "metric does not have a target."
                )

            PreviousBenchmark = GetBenchmarkName(PreviousMetric)

            ActualExpression = BuildMetricValueExpression(
                ValueType="ytd",
                MeasureReference=GetMeasureReference(PreviousMetric["Measure"]),
                BenchmarkName=PreviousBenchmark,
            )

            TargetExpression = BuildMetricValueExpression(
                ValueType="ytd",
                MeasureReference=GetMeasureReference(PreviousMetric["Target"]),
                BenchmarkName=PreviousBenchmark,
            )

            return f"({ActualExpression}) " f"- ({TargetExpression})"

        return BuildMetricValueExpression(
            ValueType="ytd",
            MeasureReference=GetMeasureReference(Metric["Measure"]),
            BenchmarkName=GetBenchmarkName(Metric),
        )

    def BuildYtdTarget(
        Index: int,
        Metric: dict[str, Any],
    ) -> str | None:
        """
        Builds a YTD target expression when enabled.
        """

        YtdTargetFlag = str(Metric["YTDTargetFlag"]).strip().lower() == "yes"

        if not YtdTargetFlag or not HasTextValue(Metric.get("Target")):
            return None

        return BuildMetricValueExpression(
            ValueType="ytd",
            MeasureReference=GetMeasureReference(Metric["Target"]),
            BenchmarkName=GetBenchmarkName(Metric),
        )

    def BuildPytd(
        Index: int,
        Metric: dict[str, Any],
    ) -> str | None:
        """
        Builds a prior-year-to-date expression when enabled.
        """

        if str(Metric["YoYFlag"]).strip().lower() != "yes":
            return None

        return BuildMetricValueExpression(
            ValueType="pytd",
            MeasureReference=GetMeasureReference(Metric["Measure"]),
            BenchmarkName=GetBenchmarkName(Metric),
        )

    # --------------------------------------------------------
    # MONTHLY ACTUAL VALUES
    # --------------------------------------------------------

    Lines.extend(
        BuildMetricSwitchVariable(
            VariableName="monthPrior2",
            MetricList=MetricList,
            ExpressionBuilder=BuildActualMonth("[Month Year Prior 2]"),
        )
    )

    Lines.extend(
        BuildMetricSwitchVariable(
            VariableName="monthPrior1",
            MetricList=MetricList,
            ExpressionBuilder=BuildActualMonth("[Month Year Prior 1]"),
        )
    )

    Lines.extend(
        BuildMetricSwitchVariable(
            VariableName="currentMonth",
            MetricList=MetricList,
            ExpressionBuilder=BuildActualMonth("[Month Year Current]"),
        )
    )

    # --------------------------------------------------------
    # MONTHLY TARGET VALUES
    # --------------------------------------------------------

    Lines.extend(
        BuildMetricSwitchVariable(
            VariableName="monthPrior2Target",
            MetricList=MetricList,
            ExpressionBuilder=BuildTargetMonth("[Month Year Prior 2]"),
        )
    )

    Lines.extend(
        BuildMetricSwitchVariable(
            VariableName="monthPrior1Target",
            MetricList=MetricList,
            ExpressionBuilder=BuildTargetMonth("[Month Year Prior 1]"),
        )
    )

    Lines.extend(
        BuildMetricSwitchVariable(
            VariableName="currentMonthTarget",
            MetricList=MetricList,
            ExpressionBuilder=BuildTargetMonth("[Month Year Current]"),
        )
    )

    # --------------------------------------------------------
    # YTD AND PYTD VALUES
    # --------------------------------------------------------

    Lines.extend(
        BuildMetricSwitchVariable(
            VariableName="ytd",
            MetricList=MetricList,
            ExpressionBuilder=BuildYtd,
        )
    )

    Lines.extend(
        BuildMetricSwitchVariable(
            VariableName="ytdTarget",
            MetricList=MetricList,
            ExpressionBuilder=BuildYtdTarget,
        )
    )

    Lines.extend(
        BuildMetricSwitchVariable(
            VariableName="pytd",
            MetricList=MetricList,
            ExpressionBuilder=BuildPytd,
        )
    )

    # --------------------------------------------------------
    # FORMAT STRING
    # --------------------------------------------------------

    Lines.extend(
        [
            "            VAR _formatString =",
            "                SWITCH(",
            "                    _metricKey,",
        ]
    )

    for Metric in MetricList:
    
        MetricKey = EscapeDaxText(
            Metric["MetricKey"]
        )
    
        FormatString = str(
            Metric["FormatString"]
        ).strip()
    
   
        FormatString = EscapeDaxText(
            FormatString
        )
    
        Lines.append(
            f'                    "{MetricKey}", "{FormatString}",'
        )

    Lines.extend(
        [
            '                    "#,##0"',
            "                )",
            "",
        ]
    )

    # --------------------------------------------------------
    # PERCENT FLAG
    # --------------------------------------------------------
    
    Lines.extend(
        [
            "            VAR _isPercent =",
            "                SWITCH(",
            "                    _metricKey,",
        ]
    )
    
    for Metric in MetricList:
    
        MetricKey = EscapeDaxText(
            Metric["MetricKey"]
        )
    
        IsPercent = (
            "%" in str(
                Metric["FormatString"]
            )
        )
    
        Lines.append(
            f'                    "{MetricKey}", '
            f'{"TRUE()" if IsPercent else "FALSE()"},'
        )
    
    Lines.extend(
        [
            "                    FALSE()",
            "                )",
            "",
        ]
    )

    # --------------------------------------------------------
    # MONTH TARGET DISPLAY FLAG
    # --------------------------------------------------------

    Lines.extend(
        [
            "            VAR _monthTargetDisplay =",
            "                SWITCH(",
            "                    _metricKey,",
        ]
    )

    for Metric in MetricList:
        MetricKey = EscapeDaxText(Metric["MetricKey"])

        MonthTargetDisplayFlag = (
            str(Metric["MonthTargetDisplayFlag"]).strip().lower() == "yes"
        )

        DaxFlag = "TRUE()" if MonthTargetDisplayFlag else "FALSE()"

        Lines.append(f'                    "{MetricKey}", ' f"{DaxFlag},")

    Lines.extend(
        [
            "                    FALSE()",
            "                )",
            "",
        ]
    )

    # --------------------------------------------------------
    # YTD COMPARISON AND TREND VALUES
    # --------------------------------------------------------

    Lines.extend(
        [
            "            VAR _ytdRatio =",
            "                DIVIDE(",
            "                    _ytd,",
            "                    _ytdTarget",
            "                )",
            "",
            "            VAR _variance =",
            "                _ytd - _pytd",
            "",
            "            VAR _variancePct =",
            "                DIVIDE(",
            "                    _variance,",
            "                    _pytd",
            "                )",
            "",
            "            VAR _trendIcon =",
            "                SWITCH(",
            "                    TRUE(),",
            "                    ISBLANK(_variance), BLANK(),",
            '                    _variance > 0, "up",',
            '                    _variance < 0, "down",',
            "                    BLANK()",
            "                )",
            "",
        ]
    )

    # --------------------------------------------------------
    # METRIC STATUS
    # --------------------------------------------------------

    Lines.extend(
        [
            "            VAR _status =",
            "                SWITCH(",
            "                    _metricKey,",
        ]
    )
#    print("\nSTATUS DEBUG")

    for Metric in MetricList:
        ThresholdFlag = str(Metric["ThresholdFlag"]).strip().lower() == "yes"

        if not ThresholdFlag:
            continue

        MetricKey = EscapeDaxText(Metric["MetricKey"])

        RedOperator = str(Metric["RedOperator"]).strip()

        RedThreshold = FormatDaxNumber(Metric["RedThreshold"])

        AmberOperator = str(Metric["AmberOperator"]).strip()

        AmberThreshold = FormatDaxNumber(Metric["AmberThreshold"])

        YtdTargetFlag = str(Metric["YTDTargetFlag"]).strip().lower() == "yes"

        MonthTargetFlag = str(Metric["MonthTargetFlag"]).strip().lower() == "yes"

#        print(
#            MetricKey,        
#            ThresholdFlag,         
#            YtdTargetFlag,         
#           MonthTargetFlag,     
#        )

        if YtdTargetFlag:
            ValueExpression = "_ytdRatio"

            BlankExpression = "ISBLANK(_ytd) " "|| ISBLANK(_ytdTarget)"

        elif MonthTargetFlag:
            ValueExpression = "DIVIDE(" "_currentMonth, " "_currentMonthTarget" ")"

            BlankExpression = (
                "ISBLANK(_currentMonth) " "|| ISBLANK(_currentMonthTarget)"
            )

        else:
            continue

        Lines.extend(
            [
                f'                    "{MetricKey}",',
                "                        SWITCH(",
                "                            TRUE(),",
                ("                            " f"{BlankExpression}, BLANK(),"),
                (
                    "                            "
                    f"{ValueExpression} "
                    f"{RedOperator} "
                    f"{RedThreshold}, "
                    '"red",'
                ),
                (
                    "                            "
                    f"{ValueExpression} "
                    f"{AmberOperator} "
                    f"{AmberThreshold}, "
                    '"amber",'
                ),
                '                            "green"',
                "                        ),",
            ]
        )

    Lines.extend(
        [
            "                    BLANK()",
            "                )",
            "",
        ]
    )

    # --------------------------------------------------------
    # YTD HTML
    # --------------------------------------------------------

    Lines.extend(
        [
            "            VAR _ytdHtml =",
            "                IF(",
            "                    ISBLANK(_ytd),",
            '                    "",',
            "                    FORMAT(",
            "                        _ytd,",
            "                        _formatString",
            "                    )",
            "                        & IF(",
            "                            ISBLANK(_ytdRatio),",
            '                            "",',
            '                            " ("',
            "                                & FORMAT(",
            "                                    _ytdRatio,",
            '                                    "0%"',
            "                                )",
            '                                & ")"',
            "                        )",
            "                )",
            "",
        ]
    )
    
    # --------------------------------------------------------
    # MONTHLY HTML
    # --------------------------------------------------------
    
    for (
        ValueName,
        TargetName,
        HtmlName,
    ) in (
        (
            "monthPrior2",
            "monthPrior2Target",
            "monthPrior2Html",
        ),
        (
            "monthPrior1",
            "monthPrior1Target",
            "monthPrior1Html",
        ),
        (
            "currentMonth",
            "currentMonthTarget",
            "currentMonthHtml",
        ),
    ):
        Lines.extend(
            [
                f"            VAR _{HtmlName} =",
                "                IF(",
                f"                    NOT ISBLANK(_{ValueName}),",
                "                    IF(",
                f"                        _monthTargetDisplay",
                f"                            && NOT ISBLANK(_{TargetName}),",
    
                "                        IF(",
                "                            _isPercent,",
    
                "                            FORMAT(",
                f"                                _{ValueName},",
                '                                SUBSTITUTE(_formatString, "%", "")',
                "                            )",
                '                                & "%"',
    
                "                            ,",
    
                "                            FORMAT(",
                f"                                _{ValueName},",
                "                                _formatString",
                "                            )",
    
                "                        )",
    
                '                            & " ("',
                "                            & FORMAT(",
                "                                DIVIDE(",
                f"                                    _{ValueName},",
                f"                                    _{TargetName}",
                "                                ),",
                '                                "0%"',
                "                            )",
                '                            & ")",',
    
                "                        IF(",
                "                            _isPercent,",
    
                "                            FORMAT(",
                f"                                _{ValueName},",
                '                                SUBSTITUTE(_formatString, "%", "")',
                "                            )",
                '                                & "%"',
    
                "                            ,",
    
                "                            FORMAT(",
                f"                                _{ValueName},",
                "                                _formatString",
                "                            )",
    
                "                        )",
    
                "                    ),",
                '                    ""',
                "                )",
                "",
            ]
        )

    # --------------------------------------------------------
    # TREND HTML
    # --------------------------------------------------------

    Lines.extend(
        [
            "            VAR _trendHtml =",
            "                SWITCH(",
            "                    TRUE(),",
            '                    ISBLANK(_variancePct), "",',
            '                    _trendIcon = "up",',
            ("                        " "\"<span class='trend up'>▲ \""),
            ("                            " '& FORMAT(_variancePct, "0.0%")'),
            ("                            " '& "</span>",'),
            '                    _trendIcon = "down",',
            ("                        " "\"<span class='trend down'>▼ \""),
            ("                            " '& FORMAT(ABS(_variancePct), "0.0%")'),
            ("                            " '& "</span>",'),
            ("                    " 'FORMAT(_variancePct, "0.0%")'),
            "                )",
            "",
        ]
    )

    # --------------------------------------------------------
    # ROW HTML
    # --------------------------------------------------------

    Lines.extend(
        [
            "            VAR _rowHtml =",
            '                "<tr>"',
            (
                '                    & "<td class='
                "'label'>\""
                " & [MetricName]"
                ' & "</td>"'
            ),
            ('                    & "<td class=' "'status'><span class='dot \""),
            ("                    & COALESCE(" "_status, " '""' ")"),
            ("                    & " '"\'></span></td>"'),
            (
                '                    & "<td class='
                "'number'>\""
                " & _ytdHtml"
                ' & "</td>"'
            ),
            (
                '                    & "<td class='
                "'number'>\""
                " & _monthPrior2Html"
                ' & "</td>"'
            ),
            (
                '                    & "<td class='
                "'number'>\""
                " & _monthPrior1Html"
                ' & "</td>"'
            ),
            (
                '                    & "<td class='
                "'number'>\""
                " & _currentMonthHtml"
                ' & "</td>"'
            ),
            (
                '                    & "<td class='
                "'number'>\""
                " & _trendHtml"
                ' & "</td>"'
            ),
            '                    & "</tr>"',
            "",
            "            RETURN",
            "                ROW(",
            '                    "RowHtml", _rowHtml',
            "                )",
            "    )",
        ]
    )

    return "\n".join(Lines)


# ============================================================
# DYNAMIC HTML CREATION
# ============================================================
def BuildOperationalRows(TableName: str, MetricList: list[dict[str, Any]]) -> str:
    """Builds project-aware HTML from one generated virtual metric table."""
    MetricTable = BuildMetricTable(MetricList=MetricList)

    CommentPanelHtml = """
         <td class='right commentbackground'>
           <div class='commentsheader'></div>
           <div class='commentary'></div>
          </td>
        """

    if IncludeCommentPanel:
        CommentPanelHtml = """
             <td class='right commentbackground'>
               <div class='commentsheader'>Performance Summary</div>
               <div class='commentary'>" & _performanceTable & "</div>
              </td>
            """

    EscapedTableName = EscapeDaxText(TableName)

    return (
        f"{BuildSectionHeader('DYNAMIC HTML CREATION')}\n"
        "VAR _selectedProject =\n"
        "    SELECTEDVALUE(\n"
        "        'Projects'[Project Name]\n"
        "    )\n\n"
        "VAR _hasSpecificMetrics =\n"
        "    COUNTROWS(\n"
        "        FILTER(\n"
        f"            ALL('{EscapedTableName}'),\n"
        "            [ProjectName] = _selectedProject\n"
        "        )\n"
        "    ) > 0\n\n"
        "VAR _projectMetrics =\n"
        "    FILTER(\n"
        f"        ALL('{EscapedTableName}'),\n"
        "        [ProjectName] = _selectedProject\n"
        "            || (\n"
        '                [ProjectName] = "ALL"\n'
        "                    && NOT _hasSpecificMetrics\n"
        "            )\n"
        "    )\n\n"
        f"{MetricTable}\n\n"
        "VAR _operationalRows =\n"
        "    CONCATENATEX(\n"
        "        _metricTable,\n"
        "        [RowHtml],\n"
        '        "",\n'
        "        [SortOrder],\n"
        "        ASC\n"
        "    )\n\n"
        "RETURN\n"
        "    IF(\n"
        "        NOT HASONEVALUE(\n"
        "            'Projects'[Project Name]\n"
        "        )\n"
        "            || ISEMPTY(_projectMetrics),\n"
        "        BLANK(),\n"
        '        "\n'
        "    <!-- OPERATIONAL EXCELLENCE -->\n"
        "    <tr>\n"
        "      <td colspan='6' class='header'>\n"
        "        Operational Excellence\n"
        "      </td>\n"
        "    </tr>\n"
        "    <tr>\n"
        "      <td class='left'>\n"
        "        <table class='metric'>\n"
        "          <colgroup>\n"
        "            <col style='width:34%'>\n"
        "            <col style='width:3%'>\n"
        "            <col style='width:12.6%'>\n"
        "            <col style='width:12.6%'>\n"
        "            <col style='width:12.6%'>\n"
        "            <col style='width:12.6%'>\n"
        "            <col style='width:12.6%'>\n"
        "          </colgroup>\n"
        "          <tr class='subheader'>\n"
        "            <th></th>\n"
        "            <th></th>\n"
        '            <th>PY \'" & FORMAT(RIGHT([Current Fiscal Year], 2), "##") & " YTD</th>\n'
        '            <th>" & [Month Display Prior 2] & "</th>\n'
        '            <th>" & [Month Display Prior 1] & "</th>\n'
        '            <th>" & [Month Display Current] & "</th>\n'
        '            <th>" & _trending & "</th>\n'
        '          </tr>"\n'
        "        & _operationalRows\n"
        '        & "\n'
        "        </table>\n"
        "      </td>\n"
        f"{CommentPanelHtml}\n"
        "    </tr>\n"
        '        "\n'
        "    )"
    )


# ============================================================
# FULL DAX GENERATOR
# ============================================================


def GenerateDax(
    TableName: str,
    MetricList: list[dict[str, Any]],
    IncludeLayoutTable: bool = True,
    ReturnBlankStatus: bool = True,
    IndentLevel: int = 0,
) -> str:
    ValidateMetrics(MetricList)
    SortedMetrics = sorted(
        MetricList,
        key=lambda Metric: (
            str(Metric["ProjectName"]).strip().upper(),
            Metric["SortOrder"],
        ),
    )
    OutputSections = []

    if IncludeLayoutTable:
        OutputSections.extend(
            [
                BuildSectionHeader("METRIC LAYOUT TABLE"),
                BuildLayoutTable(TableName=TableName, MetricList=SortedMetrics),
            ]
        )

    OutputSections.extend(
        [
            BuildSectionHeader("TREND COLUMN VARIABLE"),
            f'VAR _trending = "{TrendingLabel}"',
        ]
    )

    if ProjectType == "WIOA":
        OutputSections.extend(
            [
                BuildWioaCommentSection(),
                BuildWioaCommentStatus(),
                BuildWioaPerformanceTable(),
            ]
        )

    OutputSections.append(
        BuildOperationalRows(
            TableName=TableName,
            MetricList=SortedMetrics,
        )
    )

    GeneratedDax = "\n\n".join(OutputSections)
    return GetIndentedText(GeneratedDax.strip(), IndentLevel)


# ============================================================
# CLIPBOARD FUNCTION
# ============================================================


def CopyTextToClipboard(Text: str) -> bool:
    try:
        import pyperclip

        pyperclip.copy(Text)
        if pyperclip.paste() != Text:
            print("Warning: Clipboard verification did not match the generated DAX.")
            return False
        return True
    except ImportError:
        print("Clipboard copy skipped because pyperclip is not installed.")
        print("Install it in Jupyter with: %pip install pyperclip")
        return False
    except Exception as ClipboardError:
        print("Clipboard copy failed, but the DAX file was still created.")
        print(f"Clipboard error: {ClipboardError}")
        return False


# ============================================================
# GENERATE, SAVE, COPY, AND DISPLAY
# ============================================================

try:
    DaxCode = GenerateDax(
        TableName=LayoutTableName,
        MetricList=Metrics,
        IncludeLayoutTable=GenerateLayoutTable,
        ReturnBlankStatus=BlankStatusWhenValueIsBlank,
        IndentLevel=BaseIndent,
    )

    OutputPath = Path(OutputFileName).resolve()
    OutputPath.write_text(DaxCode, encoding="utf-8")

    ClipboardCopied = CopyTextToClipboard(DaxCode) if CopyToClipboard else False

    print("=" * 70)
    print("DAX GENERATION COMPLETE")
    print("=" * 70)
    print(f"Metric count: {len(Metrics)}")
    print(f"Layout table: {LayoutTableName}")
    print(f"Saved file: {OutputPath}")

    if CopyToClipboard:
        print(
            "Clipboard: DAX copied successfully"
            if ClipboardCopied
            else "Clipboard: DAX was not copied"
        )
    else:
        print("Clipboard: Disabled in configuration")

    print("=" * 70)
    print()
    print(DaxCode)

except Exception as GenerationError:
    print("=" * 70)
    print("DAX GENERATION FAILED")
    print("=" * 70)
    print(str(GenerationError))
    raise


DAX GENERATION COMPLETE
Metric count: 5
Layout table: ARG Operational Metric Layout
Saved file: C:\Users\StevenFoster\Foster-BI-Services\Power-BI-Portfolio\Python\DAX-HTML-Operational-Excellence\scripts\ARG_Operational_Metric_DAX.txt
Clipboard: DAX copied successfully

/* ============================================================
   METRIC LAYOUT TABLE
   ============================================================ */

ARG Operational Metric Layout =
DATATABLE(
    "ProjectName", STRING,
    "SortOrder", INTEGER,
    "MetricName", STRING,
    "MetricKey", STRING,
{
    {"ALL", 19, "CO Determinations (% Target)", "_dCOdeterminations"},
    {"ALL", 20, "CO TAT (Application Review)", "_dCOtatApplicationReview"},
    {"ALL", 21, "CO Avg Days Ahead of Target", "_dCOavgDaysAheadOfTarget"},
    {"ALL", 22, "PA TAT (Application Review)", "_dPAtatApplicationReview"},
    {"ALL", 23, "PA Avg Days Ahead of Target", "_dPAavgDaysAheadOfTarget"}
}
)

/* ============================================